In [4]:
# decision dans l'incertain avec tkinter
# criteres : maxmin maxmax hurwicz laplace bernoulli minimax regret

import tkinter as tk
from tkinter import messagebox
import numpy as np

# couleurs theme sombre
BG       = "#0a0a0a"
PANEL    = "#111111"
YELLOW   = "#FFD600"
RED      = "#E53935"
RED_DARK = "#B71C1C"
WHITE    = "#F5F5F5"
GRAY     = "#2a2a2a"
GREEN    = "#66BB6A"

# fonts courier pour style technique
FONT_TITLE = ("Courier New", 22, "bold")
FONT_SUB   = ("Courier New", 12)
FONT_HEAD  = ("Courier New", 13, "bold")
FONT_BODY  = ("Courier New", 13)
FONT_BTN   = ("Courier New", 14, "bold")
FONT_RES   = ("Courier New", 13, "bold")

# chemin du fichier de donnees
FILE_PATH = "C:/Users/skand/OneDrive/Documents/quatre_criteres_decis_ds_incertain.txt"

# etats du monde
ETATS = ["Croissance", "Stagnation", "Récession"]

# decisions affichees par defaut au lancement
DEFAULT_DECISIONS = [
    "Investir dans la R&D",
    "Augmenter la production",
    "Réduire les coûts",
    "Campagne marketing",
]

# liste des criteres disponibles
ALL_CRITERIA = ["Maxmin", "Maxmax", "Hurwicz", "Laplace", "Bernoulli", "MiniMax Regret"]


# -----------------------------
# critères
# -----------------------------
def maxmin(matrix):
    mins = np.min(matrix, axis=1)
    idx  = int(np.argmax(mins))
    return idx, mins[idx]


def maxmax(matrix):
    maxs = np.max(matrix, axis=1)
    idx  = int(np.argmax(maxs))
    return idx, maxs[idx]


def hurwicz(matrix, alpha):
    scores = alpha * np.max(matrix, axis=1) + (1 - alpha) * np.min(matrix, axis=1)
    idx    = int(np.argmax(scores))
    return idx, scores[idx]


def laplace(matrix):
    means = np.mean(matrix, axis=1)
    idx   = int(np.argmax(means))
    return idx, means[idx]


def bernoulli(matrix):
    if np.any(matrix <= 0):
        return None, None
    log_matrix = np.log(matrix)
    means      = np.mean(log_matrix, axis=1)
    idx        = int(np.argmax(means))
    return idx, means[idx]


def minimax_regret(matrix):
    col_max = np.max(matrix, axis=0)
    regret_mat = col_max - matrix
    max_regrets = np.max(regret_mat, axis=1)
    idx = int(np.argmin(max_regrets))
    return idx, max_regrets[idx]


# -----------------------------
# helpers UI
# -----------------------------
def make_button(parent, text, command, bg=RED, fg=WHITE):
    btn = tk.Button(
        parent,
        text=text,
        command=command,
        font=FONT_BTN,
        bg=bg,
        fg=fg,
        activebackground=RED_DARK,
        activeforeground=WHITE,
        relief="flat",
        bd=0,
        padx=24,
        pady=10,
        cursor="hand2"
    )
    hover = RED_DARK if bg == RED else "#c9a800"
    btn.bind("<Enter>", lambda e: btn.config(bg=hover))
    btn.bind("<Leave>", lambda e: btn.config(bg=bg))
    return btn


def hsep(parent, row, colspan):
    tk.Frame(parent, bg=GRAY, height=1).grid(
        row=row, column=0, columnspan=colspan,
        sticky="ew", padx=20, pady=6
    )


def fmt_value(x):
    if abs(x - int(x)) < 1e-9:
        return str(int(x))
    return f"{x:.2f}"


# -----------------------------
# arbre de décision
# -----------------------------
def draw_tree(canvas, decisions, states, matrix, chosen_idx, criterion, score_value):
    canvas.delete("all")
    canvas.update_idletasks()

    w = max(canvas.winfo_width(), 1080)
    h = 460
    canvas.config(height=h)

    root_x, root_y = 70, h // 2
    decision_x = 330
    state_x = 720
    value_x = 990

    # racine
    canvas.create_rectangle(
        root_x - 18, root_y - 18, root_x + 18, root_y + 18,
        fill=YELLOW, outline=RED, width=2
    )
    canvas.create_text(root_x, root_y, text="D", font=FONT_HEAD, fill=BG)

    dec_y_positions = [85, 180, 275, 370]

    for i, dec_y in enumerate(dec_y_positions):
        is_best = (i == chosen_idx)
        line_color = YELLOW if is_best else "#8a8a8a"
        text_color = YELLOW if is_best else WHITE
        node_outline = RED if is_best else GRAY
        node_fill = "#1d1d1d" if is_best else PANEL

        # branche principale
        canvas.create_line(
            root_x + 18, root_y, decision_x - 30, dec_y,
            fill=line_color, width=2 if is_best else 1
        )

        # texte décision
        canvas.create_text(
            decision_x - 55, dec_y - 14,
            text=decisions[i],
            font=FONT_BODY,
            fill=text_color,
            anchor="w"
        )

        # noeud état
        canvas.create_oval(
            decision_x - 18, dec_y - 18, decision_x + 18, dec_y + 18,
            fill=node_fill, outline=node_outline, width=2
        )
        canvas.create_text(decision_x, dec_y, text="M", font=FONT_HEAD, fill=text_color)

        # trois états
        state_y_positions = [dec_y - 26, dec_y, dec_y + 26]

        for j, sy in enumerate(state_y_positions):
            val = matrix[i, j]
            vcolor = GREEN if val > 0 else (RED if val < 0 else WHITE)

            canvas.create_line(
                decision_x + 18, dec_y, state_x - 80, sy,
                fill=line_color if is_best else "#777777",
                width=2 if is_best else 1
            )

            canvas.create_text(
                state_x - 75, sy,
                text=states[j],
                font=FONT_BODY,
                fill=WHITE,
                anchor="w"
            )

            canvas.create_line(
                state_x + 10, sy, value_x - 40, sy,
                fill="#999999",
                width=1
            )

            canvas.create_text(
                value_x, sy,
                text=fmt_value(val),
                font=FONT_BODY,
                fill=vcolor,
                anchor="e"
            )

    # résumé
    summary = (
        f"Critère : {criterion}   |   "
        f"Décision retenue : {decisions[chosen_idx]}   |   "
        f"Score : {fmt_value(score_value)}"
    )
    canvas.create_text(w // 2, h - 20, text=summary, font=FONT_SUB, fill=YELLOW)


# -----------------------------
# interface principale
# -----------------------------
def decision_interface():
    root = tk.Tk()
    root.title("Décision dans l'incertain")
    root.configure(bg=BG)
    root.state("zoomed")

    # zone scrollable principale
    container = tk.Frame(root, bg=BG)
    container.pack(fill="both", expand=True)

    main_canvas = tk.Canvas(container, bg=BG, highlightthickness=0)
    scrollbar = tk.Scrollbar(container, orient="vertical", command=main_canvas.yview)
    main_canvas.configure(yscrollcommand=scrollbar.set)

    scrollbar.pack(side="right", fill="y")
    main_canvas.pack(side="left", fill="both", expand=True)

    outer = tk.Frame(main_canvas, bg=BG)
    window_id = main_canvas.create_window((0, 0), window=outer, anchor="n")

    def update_scrollregion(event=None):
        main_canvas.configure(scrollregion=main_canvas.bbox("all"))

    def on_canvas_configure(event):
        canvas_width = event.width
        main_canvas.itemconfig(window_id, width=canvas_width)

    outer.bind("<Configure>", update_scrollregion)
    main_canvas.bind("<Configure>", on_canvas_configure)

    def on_mousewheel(event):
        main_canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")

    main_canvas.bind_all("<MouseWheel>", on_mousewheel)

    COLS = 7

    criterion_var = tk.StringVar(value="Maxmin")
    alpha_var     = tk.DoubleVar(value=0.5)
    result_var    = tk.StringVar(value="?")

    matrix_entries   = {}
    decision_entries = []

    # pour centrer la grille
    for c in range(COLS):
        outer.grid_columnconfigure(c, weight=1)

    # titre principal
    tk.Label(
        outer,
        text="  APPLICATION DES SIX CRITÈRES DE DÉCISION DANS L'INCERTAIN",
        font=FONT_TITLE, bg=BG, fg=YELLOW
    ).grid(row=0, column=0, columnspan=COLS, pady=(28, 4))

    tk.Label(
        outer,
        text="Maxmin  ·  Maxmax  ·  Hurwicz  ·  Laplace  ·  Bernoulli  ·  MiniMax Regret",
        font=FONT_SUB, bg=BG, fg=RED
    ).grid(row=1, column=0, columnspan=COLS, pady=(0, 10))

    hsep(outer, 2, COLS)

    # bouton charger
    btn_load = make_button(outer, "  Charger le fichier txt", lambda: None, bg=YELLOW, fg=BG)
    btn_load.grid(row=3, column=0, columnspan=COLS, pady=(8, 14))

    hsep(outer, 4, COLS)

    # en-têtes tableau
    tk.Label(
        outer, text="Décision  \\  État", font=FONT_HEAD,
        bg=BG, fg=YELLOW, width=28, anchor="w"
    ).grid(row=5, column=0, padx=(20, 6), pady=6)

    header_colors = [YELLOW, RED, WHITE]
    for j, etat in enumerate(ETATS):
        tk.Label(
            outer, text=etat, font=FONT_HEAD, bg=BG, fg=header_colors[j],
            width=14, anchor="center"
        ).grid(row=5, column=j+1, padx=10, pady=6)

    tk.Label(outer, text="", bg=BG, width=3).grid(row=5, column=4)
    tk.Label(outer, text="", bg=BG, width=3).grid(row=5, column=5)
    tk.Label(outer, text="", bg=BG, width=3).grid(row=5, column=6)

    # lignes matrice
    for i in range(4):
        d = tk.Entry(
            outer, font=FONT_BODY, bg=PANEL, fg=YELLOW,
            insertbackground=YELLOW, relief="flat",
            highlightthickness=1, highlightcolor=RED,
            highlightbackground=GRAY, width=28, justify="left"
        )
        d.insert(0, DEFAULT_DECISIONS[i])
        d.grid(row=6+i, column=0, padx=(20, 6), pady=6)
        decision_entries.append(d)

        for j in range(3):
            e = tk.Entry(
                outer, font=FONT_BODY, bg=PANEL, fg=WHITE,
                insertbackground=WHITE, relief="flat",
                highlightthickness=1, highlightcolor=RED,
                highlightbackground=GRAY, width=14, justify="center"
            )
            e.grid(row=6+i, column=j+1, padx=10, pady=6)
            matrix_entries[(i, j)] = e

    hsep(outer, 10, COLS)

    # Hurwicz
    tk.Label(
        outer, text="Alpha — Hurwicz :", font=FONT_HEAD,
        bg=BG, fg=YELLOW
    ).grid(row=11, column=0, sticky="w", padx=20, pady=8)

    alpha_lbl = tk.Label(
        outer, text="α = 0.50",
        font=("Courier New", 13, "bold"),
        bg=BG, fg=RED, width=8
    )
    alpha_lbl.grid(row=11, column=2, padx=6)

    def on_alpha(val):
        alpha_lbl.config(text=f"α = {float(val):.2f}")

    tk.Scale(
        outer, variable=alpha_var, from_=0.0, to=1.0,
        orient="horizontal", resolution=0.01, command=on_alpha,
        bg=BG, fg=YELLOW, troughcolor=GRAY,
        highlightthickness=0, sliderlength=26,
        activebackground=RED, length=320, showvalue=False
    ).grid(row=11, column=1, padx=6, pady=6)

    hsep(outer, 12, COLS)

    # critères
    tk.Label(
        outer, text="Critère :", font=FONT_HEAD,
        bg=BG, fg=YELLOW
    ).grid(row=13, column=0, sticky="w", padx=20, pady=4)

    for k, opt in enumerate(ALL_CRITERIA[:3]):
        tk.Radiobutton(
            outer, text=opt, variable=criterion_var, value=opt,
            font=FONT_BODY, bg=BG, fg=WHITE,
            selectcolor=RED, activebackground=BG,
            activeforeground=YELLOW
        ).grid(row=13, column=k+1, padx=6, pady=4)

    tk.Label(outer, text="", bg=BG).grid(row=14, column=0)
    for k, opt in enumerate(ALL_CRITERIA[3:]):
        tk.Radiobutton(
            outer, text=opt, variable=criterion_var, value=opt,
            font=FONT_BODY, bg=BG, fg=WHITE,
            selectcolor=RED, activebackground=BG,
            activeforeground=YELLOW
        ).grid(row=14, column=k+1, padx=6, pady=4)

    hsep(outer, 15, COLS)

    # zone résultat
    res_frame = tk.Frame(
        outer, bg=PANEL,
        highlightthickness=2, highlightbackground=RED
    )
    res_frame.grid(
        row=16, column=0, columnspan=COLS,
        padx=20, pady=(10, 8), sticky="ew"
    )

    tk.Label(
        res_frame, text="RÉSULTAT",
        font=("Courier New", 13, "bold"),
        bg=PANEL, fg=RED
    ).pack(pady=(12, 4))

    result_lbl_holder = []
    rlbl = tk.Label(
        res_frame,
        textvariable=result_var,
        font=FONT_RES,
        bg=PANEL,
        fg=YELLOW,
        wraplength=1100,
        justify="center"
    )
    rlbl.pack(pady=(2, 14))
    result_lbl_holder.append(rlbl)

    # bouton calculer
    make_button(outer, "  Calculer", lambda: calculate()).grid(
        row=17, column=0, columnspan=COLS, pady=(10, 14)
    )

    # zone arbre
    tree_frame = tk.Frame(
        outer, bg=PANEL,
        highlightthickness=2, highlightbackground=GRAY
    )
    tree_frame.grid(
        row=18, column=0, columnspan=COLS,
        padx=20, pady=(4, 30), sticky="ew"
    )

    tk.Label(
        tree_frame, text="ARBRE DE DÉCISION",
        font=("Courier New", 13, "bold"),
        bg=PANEL, fg=RED
    ).pack(pady=(12, 6))

    tree_canvas = tk.Canvas(
        tree_frame,
        bg=PANEL,
        highlightthickness=0,
        width=1120,
        height=460
    )
    tree_canvas.pack(padx=12, pady=(0, 12), fill="both", expand=True)

    def set_result(text, color=YELLOW):
        result_var.set(text)
        result_lbl_holder[0].config(fg=color)

    # chargement fichier
    def load_file():
        try:
            with open(FILE_PATH, "r", encoding="utf-8") as f:
                lines = [l.strip() for l in f if l.strip()]

            if len(lines) < 4:
                raise ValueError("Le fichier doit contenir au moins 4 lignes.")

            for i, line in enumerate(lines[:4]):
                parts = line.split(",")
                if len(parts) < 4:
                    raise ValueError(f"Ligne {i+1} mal formée.")

                decision_entries[i].delete(0, tk.END)
                decision_entries[i].insert(0, parts[0].strip())

                for j in range(3):
                    matrix_entries[(i, j)].delete(0, tk.END)
                    matrix_entries[(i, j)].insert(0, parts[j+1].strip())

            set_result("Fichier chargé avec succès. Tu peux maintenant sélectionner un critère puis lancer le calcul.", GREEN)
            root.after(2500, lambda: result_lbl_holder[0].config(fg=YELLOW))

        except Exception as e:
            messagebox.showerror("Erreur de chargement", str(e))

    btn_load.config(command=load_file)

    # calcul
    def calculate():
        try:
            matrix = np.array([
                [float(matrix_entries[(i, j)].get()) for j in range(3)]
                for i in range(4)
            ])
        except ValueError:
            messagebox.showerror(
                "Saisie invalide",
                "Merci de remplir toutes les cases avec des nombres valides."
            )
            return

        criterion = criterion_var.get()
        alpha = alpha_var.get()
        decisions = [entry.get().strip() for entry in decision_entries]

        if criterion == "Maxmin":
            idx, val = maxmin(matrix)
            explanation = (
                f"Avec le critère Maxmin, on regarde pour chaque décision son résultat le plus défavorable. "
                f"Ensuite, on choisit celle dont le pire cas reste le plus acceptable.\n"
                f"Ici, la décision qui ressort est « {decisions[idx]} », avec une valeur de {fmt_value(val)}."
            )

        elif criterion == "Maxmax":
            idx, val = maxmax(matrix)
            explanation = (
                f"Avec le critère Maxmax, on cherche la décision qui peut donner le meilleur résultat possible.\n"
                f"Ici, l’option la plus avantageuse est « {decisions[idx]} », car elle peut atteindre {fmt_value(val)}."
            )

        elif criterion == "Hurwicz":
            idx, val = hurwicz(matrix, alpha)
            explanation = (
                f"Avec le critère de Hurwicz, on fait un compromis entre prudence et optimisme avec α = {alpha:.2f}.\n"
                f"D’après le calcul, la décision à privilégier est « {decisions[idx]} », avec un score de {fmt_value(val)}."
            )

        elif criterion == "Laplace":
            idx, val = laplace(matrix)
            explanation = (
                f"Avec le critère de Laplace, on suppose que les différents états ont la même probabilité. "
                f"On compare donc les moyennes.\n"
                f"Ici, la décision la plus intéressante est « {decisions[idx]} », avec une moyenne de {fmt_value(val)}."
            )

        elif criterion == "Bernoulli":
            if np.any(matrix <= 0):
                messagebox.showerror(
                    "Bernoulli impossible",
                    "Le critère de Bernoulli utilise ln(x).\n"
                    "Toutes les valeurs doivent être strictement positives."
                )
                return
            idx, val = bernoulli(matrix)
            explanation = (
                f"Avec le critère de Bernoulli, on utilise l’utilité logarithmique pour comparer les décisions.\n"
                f"Ici, la décision retenue est « {decisions[idx]} », avec une utilité moyenne de {fmt_value(val)}."
            )

        elif criterion == "MiniMax Regret":
            idx, val = minimax_regret(matrix)
            explanation = (
                f"Avec le critère MiniMax Regret, on cherche la décision qui limite le regret dans le pire des cas.\n"
                f"Ici, la décision la plus prudente est « {decisions[idx]} », avec un regret maximal de {fmt_value(val)}."
            )
        else:
            return

        set_result(explanation, YELLOW)

        draw_tree(
            tree_canvas,
            decisions=decisions,
            states=ETATS,
            matrix=matrix,
            chosen_idx=idx,
            criterion=criterion,
            score_value=val
        )

    root.mainloop()


if __name__ == "__main__":
    decision_interface()

Version sans arbres :

In [5]:
# decision dans l'incertain avec tkinter
# criteres : maxmin maxmax hurwicz laplace bernoulli minimax regret

import tkinter as tk
from tkinter import messagebox
import numpy as np

# couleurs theme sombre
BG       = "#0a0a0a"
PANEL    = "#111111"
YELLOW   = "#FFD600"
RED      = "#E53935"
RED_DARK = "#B71C1C"
WHITE    = "#F5F5F5"
GRAY     = "#2a2a2a"
GREEN    = "#66BB6A"

# fonts courier pour style technique
FONT_TITLE = ("Courier New", 22, "bold")
FONT_SUB   = ("Courier New", 12)
FONT_HEAD  = ("Courier New", 13, "bold")
FONT_BODY  = ("Courier New", 13)
FONT_BTN   = ("Courier New", 14, "bold")
FONT_RES   = ("Courier New", 14, "bold")

# chemin du fichier de donnees
FILE_PATH = "C:/Users/skand/OneDrive/Documents/quatre_criteres_decis_ds_incertain.txt"

# etats du monde
ETATS = ["Croissance", "Stagnation", "Récession"]

# decisions affichees par defaut au lancement
DEFAULT_DECISIONS = [
    "Investir dans la R&D",
    "Augmenter la production",
    "Réduire les coûts",
    "Campagne marketing",
]

# liste des 6 criteres disponibles (ordre d'affichage dans les radio boutons)
ALL_CRITERIA = ["Maxmin", "Maxmax", "Hurwicz", "Laplace", "Bernoulli", "MiniMax Regret"]


# calcul du critere maxmin (pessimiste : on prend le pire puis on maximise)
def maxmin(matrix):
    mins = np.min(matrix, axis=1)
    idx  = int(np.argmax(mins))
    return idx, mins[idx]


# calcul du critere maxmax (optimiste : on prend le meilleur puis on maximise)
def maxmax(matrix):
    maxs = np.max(matrix, axis=1)
    idx  = int(np.argmax(maxs))
    return idx, maxs[idx]


# critere de hurwicz : combinaison ponderee optimisme/pessimisme via alpha
def hurwicz(matrix, alpha):
    scores = alpha * np.max(matrix, axis=1) + (1 - alpha) * np.min(matrix, axis=1)
    idx    = int(np.argmax(scores))
    return idx, scores[idx]


# critere de laplace : equiprobabilite, on maximise la moyenne
def laplace(matrix):
    means = np.mean(matrix, axis=1)
    idx   = int(np.argmax(means))
    return idx, means[idx]


# critere de bernoulli : utilite logarithmique, moyenne des ln(gains)
# ne fonctionne qu avec des valeurs strictement positives
def bernoulli(matrix):
    # on verifie qu aucune valeur n est <= 0 car ln non defini
    if np.any(matrix <= 0):
        return None, None  # signale une erreur a l appelant

    log_matrix = np.log(matrix)  # ln de chaque valeur
    means      = np.mean(log_matrix, axis=1)
    idx        = int(np.argmax(means))
    return idx, means[idx]


# critere minimax regret (savage) :
# on calcule le regret de chaque decision dans chaque etat
# puis on prend le max du regret par decision et on minimise
def minimax_regret(matrix):
    # meilleure valeur par colonne (par etat)
    col_max    = np.max(matrix, axis=0)

    # regret = ecart entre le meilleur possible et ce qu on obtient
    regret_mat = col_max - matrix

    # regret maximal de chaque decision
    max_regrets = np.max(regret_mat, axis=1)

    # on choisit la decision qui minimise ce regret maximal
    idx = int(np.argmin(max_regrets))
    return idx, max_regrets[idx]


# bouton stylise avec effet de survol
def make_button(parent, text, command, bg=RED, fg=WHITE):
    btn = tk.Button(
        parent, text=text, command=command,
        font=FONT_BTN, bg=bg, fg=fg,
        activebackground=RED_DARK, activeforeground=WHITE,
        relief="flat", bd=0, padx=24, pady=10, cursor="hand2"
    )
    hover = RED_DARK if bg == RED else "#c9a800"
    btn.bind("<Enter>", lambda e: btn.config(bg=hover))
    btn.bind("<Leave>", lambda e: btn.config(bg=bg))
    return btn


# ligne separatrice horizontale (purement visuelle)
def hsep(parent, row, colspan):
    tk.Frame(parent, bg=GRAY, height=1).grid(
        row=row, column=0, columnspan=colspan,
        sticky="ew", padx=20, pady=6
    )


# fonction principale qui construit toute l interface
def decision_interface():

    root = tk.Tk()
    root.title("Décision dans l'incertain")
    root.configure(bg=BG)
    root.state("zoomed")  # plein ecran windows

    COLS = 7  # on passe a 7 colonnes pour loger 6 criteres dans les radio

    # variables tkinter partagees
    criterion_var = tk.StringVar(value="Maxmin")
    alpha_var     = tk.DoubleVar(value=0.5)
    result_var    = tk.StringVar(value="?")

    # conteneur central
    outer = tk.Frame(root, bg=BG)
    outer.place(relx=0.5, rely=0.5, anchor="center")

    matrix_entries   = {}
    decision_entries = []

    # titre principal
    tk.Label(outer,
             text="  APPLICATION DES SIX CRITÈRES DE DÉCISION DANS L'INCERTAIN",
             font=FONT_TITLE, bg=BG, fg=YELLOW).grid(
             row=0, column=0, columnspan=COLS, pady=(0, 4))

    tk.Label(outer,
             text="Maxmin  ·  Maxmax  ·  Hurwicz  ·  Laplace  ·  Bernoulli  ·  MiniMax Regret",
             font=FONT_SUB, bg=BG, fg=RED).grid(
             row=1, column=0, columnspan=COLS, pady=(0, 8))

    hsep(outer, 2, COLS)

    # bouton de chargement (command assignee plus bas apres definition de load_file)
    btn_load = make_button(outer, "  Charger le fichier txt", lambda: None,
                           bg=YELLOW, fg=BG)
    btn_load.grid(row=3, column=0, columnspan=COLS, pady=(6, 12))

    hsep(outer, 4, COLS)

    # en tetes du tableau
    tk.Label(outer, text="Décision  \\  État", font=FONT_HEAD,
             bg=BG, fg=YELLOW, width=30, anchor="w").grid(
             row=5, column=0, padx=(20, 6), pady=6)

    for j, (etat, color) in enumerate(zip(ETATS, [YELLOW, RED, WHITE])):
        tk.Label(outer, text=etat, font=FONT_HEAD, bg=BG, fg=color,
                 width=14, anchor="center").grid(row=5, column=j+1, padx=6, pady=6)

    # colonnes supplementaires vides pour equilibrer la grille
    tk.Label(outer, text="", bg=BG, width=3).grid(row=5, column=4)
    tk.Label(outer, text="", bg=BG, width=3).grid(row=5, column=5)
    tk.Label(outer, text="", bg=BG, width=3).grid(row=5, column=6)

    # lignes de la matrice (4 decisions x 3 etats)
    for i in range(4):
        d = tk.Entry(outer, font=FONT_BODY, bg=PANEL, fg=YELLOW,
                     insertbackground=YELLOW, relief="flat",
                     highlightthickness=1, highlightcolor=RED,
                     highlightbackground=GRAY, width=30, justify="left")
        d.insert(0, DEFAULT_DECISIONS[i])
        d.grid(row=6+i, column=0, padx=(20, 6), pady=6)
        decision_entries.append(d)

        for j in range(3):
            e = tk.Entry(outer, font=FONT_BODY, bg=PANEL, fg=WHITE,
                         insertbackground=WHITE, relief="flat",
                         highlightthickness=1, highlightcolor=RED,
                         highlightbackground=GRAY, width=14, justify="center")
            e.grid(row=6+i, column=j+1, padx=6, pady=6)
            matrix_entries[(i, j)] = e

    hsep(outer, 10, COLS)

    # curseur alpha pour hurwicz
    tk.Label(outer, text="Alpha — Hurwicz :", font=FONT_HEAD,
             bg=BG, fg=YELLOW).grid(row=11, column=0, sticky="w", padx=20, pady=6)

    alpha_lbl = tk.Label(outer, text="α = 0.50",
                         font=("Courier New", 13, "bold"),
                         bg=BG, fg=RED, width=8)
    alpha_lbl.grid(row=11, column=2, padx=6)

    def on_alpha(val):
        alpha_lbl.config(text=f"α = {float(val):.2f}")

    tk.Scale(outer, variable=alpha_var, from_=0.0, to=1.0,
             orient="horizontal", resolution=0.01, command=on_alpha,
             bg=BG, fg=YELLOW, troughcolor=GRAY,
             highlightthickness=0, sliderlength=26,
             activebackground=RED, length=280, showvalue=False).grid(
             row=11, column=1, padx=6, pady=6)

    hsep(outer, 12, COLS)

    # radio boutons pour les 6 criteres
    # on les dispose sur 2 lignes de 3 pour ne pas ecraser la grille
    tk.Label(outer, text="Critère :", font=FONT_HEAD,
             bg=BG, fg=YELLOW).grid(row=13, column=0, sticky="w", padx=20, pady=4)

    # premiere ligne : maxmin maxmax hurwicz
    for k, opt in enumerate(ALL_CRITERIA[:3]):
        tk.Radiobutton(outer, text=opt, variable=criterion_var, value=opt,
                       font=FONT_BODY, bg=BG, fg=WHITE,
                       selectcolor=RED, activebackground=BG,
                       activeforeground=YELLOW).grid(
                       row=13, column=k+1, padx=6, pady=4)

    # deuxieme ligne : laplace bernoulli minimax regret
    tk.Label(outer, text="", bg=BG).grid(row=14, column=0)
    for k, opt in enumerate(ALL_CRITERIA[3:]):
        tk.Radiobutton(outer, text=opt, variable=criterion_var, value=opt,
                       font=FONT_BODY, bg=BG, fg=WHITE,
                       selectcolor=RED, activebackground=BG,
                       activeforeground=YELLOW).grid(
                       row=14, column=k+1, padx=6, pady=4)

    hsep(outer, 15, COLS)

    # zone d affichage du resultat
    res_frame = tk.Frame(outer, bg=PANEL,
                         highlightthickness=2, highlightbackground=RED)
    res_frame.grid(row=16, column=0, columnspan=COLS,
                   padx=20, pady=(8, 6), sticky="ew")

    tk.Label(res_frame, text="RÉSULTAT",
             font=("Courier New", 13, "bold"),
             bg=PANEL, fg=RED).pack(pady=(10, 2))

    # on stocke le label dans une liste pour y acceder dans les closures
    result_lbl_holder = []
    rlbl = tk.Label(res_frame, textvariable=result_var,
                    font=FONT_RES, bg=PANEL, fg=YELLOW,
                    wraplength=900, justify="center")
    rlbl.pack(pady=(2, 14))
    result_lbl_holder.append(rlbl)

    # helper pour mettre a jour le resultat et sa couleur
    def set_result(text, color=YELLOW):
        result_var.set(text)
        result_lbl_holder[0].config(fg=color)

    # chargement du fichier txt et remplissage de la matrice
    def load_file():
        try:
            with open(FILE_PATH, 'r', encoding='utf-8') as f:
                lines = [l.strip() for l in f if l.strip()]

            if len(lines) < 4:
                raise ValueError("Le fichier doit contenir au moins 4 lignes.")

            for i, line in enumerate(lines[:4]):
                parts = line.split(',')
                if len(parts) < 4:
                    raise ValueError(f"Ligne {i+1} mal formée.")
                decision_entries[i].delete(0, tk.END)
                decision_entries[i].insert(0, parts[0].strip())
                for j in range(3):
                    matrix_entries[(i, j)].delete(0, tk.END)
                    matrix_entries[(i, j)].insert(0, parts[j+1].strip())

            set_result("✔  Fichier chargé avec succès.", GREEN)
            root.after(2000, lambda: result_lbl_holder[0].config(fg=YELLOW))

        except Exception as e:
            messagebox.showerror("Erreur de chargement", str(e))

    # calcul du critere selectionne et affichage du resultat
    def calculate():
        try:
            matrix = np.array([
                [float(matrix_entries[(i, j)].get()) for j in range(3)]
                for i in range(4)
            ])
        except ValueError:
            messagebox.showerror("Saisie invalide", "Toutes les cases doivent contenir des nombres.")
            return

        criterion = criterion_var.get()
        alpha     = alpha_var.get()

        # on dispatch selon le critere choisi
        if criterion == "Maxmin":
            idx, val = maxmin(matrix)
            desc = "Pessimiste — maximise le pire cas"

        elif criterion == "Maxmax":
            idx, val = maxmax(matrix)
            desc = "Optimiste — maximise le meilleur cas"

        elif criterion == "Hurwicz":
            idx, val = hurwicz(matrix, alpha)
            desc = f"Mixte — α·max + (1−α)·min   avec α = {alpha:.2f}"

        elif criterion == "Laplace":
            idx, val = laplace(matrix)
            desc = "Équiprobabilité — maximise la moyenne"

        elif criterion == "Bernoulli":
            # bernoulli necessite des valeurs strictement positives
            if np.any(matrix <= 0):
                messagebox.showerror(
                    "Bernoulli impossible",
                    "Le critère de Bernoulli utilise ln(x).\n"
                    "Toutes les valeurs de la matrice doivent être strictement positives (> 0).\n"
                    "Veuillez corriger la matrice."
                )
                return
            idx, val = bernoulli(matrix)
            desc = "Utilité logarithmique — maximise la moyenne des ln(gains)"

        elif criterion == "MiniMax Regret":
            idx, val = minimax_regret(matrix)
            desc = "Savage — minimise le regret maximal"

        else:
            return

        nom   = decision_entries[idx].get()
        extra = ""

        # pour minimax regret on precise que val est un regret (pas un gain)
        val_label = "Valeur du critère" if criterion != "MiniMax Regret" else "Regret maximal minimisé"

        set_result(
            f"Critère : {criterion}   —   {desc}\n"
            f"Meilleure décision  →  « {nom} »\n"
            f"{val_label}  →  {val:.4f}",
            YELLOW
        )

    # on connecte le vrai callback au bouton maintenant que load_file est definie
    btn_load.config(command=load_file)

    # bouton de calcul
    make_button(outer, "  Calculer", calculate).grid(
        row=17, column=0, columnspan=COLS, pady=(10, 26))

    root.mainloop()


# point d entree
if __name__ == "__main__":
    decision_interface()